# AI应用基础 · 工单M4.2《工业智能决策助手系统》 — 教师参考版
第11周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m4kb"], check=True)
print("数据已生成: data/")

## 参考实现 `m42`（含依赖导入）

In [ ]:
import csv
import json
import sys
import numpy as np
import numpy as np  # 通用科学计算


def m42():
    import re
    import requests

    BASE = "http://127.0.0.1:8000"

    def chat(msgs):
        r = requests.post(f"{BASE}/v1/chat/completions",
                          json={"messages": msgs}, timeout=5)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"]

    kb = json.load(open("data/knowledge_base.json", encoding="utf-8"))
    docs = [t for v in kb.values() for t in v]

    def retrieve(q, topk=2):
        qs = set(re.findall(r"[\u4e00-\u9fa5a-zA-Z0-9]+", q))
        return sorted(docs, key=lambda t: -len(qs & set(
            re.findall(r"[\u4e00-\u9fa5a-zA-Z0-9]+", t))))[:topk]

    qs = ["泵振动7.5mm/s怎么处理？", "轴承温度82℃要不要停机？",
          "变频器过流报警如何排查？"]
    for q in qs:
        ctx = "\n".join(retrieve(q))
        a = chat([{"role": "system", "content": "CONTEXT:" + ctx},
                  {"role": "user", "content": q}])
        print("Q:", q, "\nA:", a[:90], "\n")
    print("Agent演示: 振动7.5>4.5阈值 → 判定超标 → 检索知识库 → 生成停机检查建议。")


## 运行环境(如需)

In [ ]:
# 本地模拟LLM服务(后台线程)
import threading, sys, importlib.util
spec = importlib.util.spec_from_file_location("mock", "数据与工具/mock_llm_server.py")
mock = importlib.util.module_from_spec(spec); spec.loader.exec_module(mock)
srv = mock.ThreadingHTTPServer(("127.0.0.1", 8000), mock.Handler)
threading.Thread(target=srv.serve_forever, daemon=True).start()
print("mock LLM on :8000")

## 执行与验证

In [ ]:
results = m42()